# 04 — Hall Baseline

Forward-walking validation of the extended Hall energy-balance model with **population-default parameters**. No personalization yet — that's Phase 2.

Renders the three required plots (weight trajectory + 95% band, body-fat trajectory, residual time-series) and prints the summary metrics (MAE, calibration coverage, drift p-value).

In [1]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from body_sim.config import DEFAULT_PROFILE
from body_sim import validation, evaluate, plotting

df = pd.read_parquet(REPO_ROOT / 'notebooks' / 'predictions' / 'daily_rollup.parquet')
df.head()

,intake_kcal,protein_g,carb_g,fat_g,sodium_mg,meal_types_logged,intake_coverage,intake_logged,steps,active_kcal_fitbit,...,vigorous_min,cardio_min,weight_kg,bf_pct,n_weighins,rhr_bpm,sleep_total_h_prev_night,workout_kcal,workout_min,reference_weight_kg
date,,,,,,,,,,,,,,,,,,,,,
2026-03-20,NaN,NaN,NaN,NaN,NaN,,0.0,False,NaN,NaN,...,0,0,NaN,NaN,0,67.0,NaN,0.0,0,80.0
2026-03-21,NaN,NaN,NaN,NaN,NaN,,0.0,False,NaN,NaN,...,0,0,NaN,NaN,0,NaN,NaN,0.0,0,80.0
2026-03-22,NaN,NaN,NaN,NaN,NaN,,0.0,False,NaN,NaN,...,0,0,NaN,NaN,0,67.0,NaN,0.0,0,80.0
2026-03-23,NaN,NaN,NaN,NaN,NaN,,0.0,False,NaN,NaN,...,0,2,NaN,NaN,0,68.0,NaN,0.0,0,80.0
2026-03-24,NaN,NaN,NaN,NaN,NaN,,0.0,False,NaN,NaN,...,0,0,NaN,NaN,0,69.0,8.316667,0.0,0,80.0


In [2]:
walk = validation.forward_walk(
    df, step_days=7, profile=DEFAULT_PROFILE, sample_n=200, seed=42
)
walk.head()

,date,sample,predicted_weight_kg,observed_weight_kg,fat_mass_kg,lean_mass_kg,body_fat_pct
0,2026-05-02,0,86.213932,85.30,18.732759,66.510973,21.975527
1,2026-05-03,0,86.033270,84.75,18.738735,66.515120,21.979927
2,2026-05-04,0,85.801796,82.85,18.716708,66.499839,21.963702
3,2026-05-05,0,85.786055,NaN,18.718233,66.500898,21.964825
4,2026-05-06,0,85.789925,NaN,18.701909,66.489561,21.952796


## Summary metrics

In [3]:
rep = evaluate.summary_report(walk)
print(f'MAE:                   {rep["mae"]:.3f} kg')
print(f'95% calibration:       {100*rep["calibration_coverage"]:.1f}% of observed weigh-ins inside band')
print(f'Residual drift p-value: {rep["residual_drift_p"]:.3f}')
print(f'Observations used:      {rep["n_observations"]}')

print()
print('Phase 1 passing thresholds:')
print(f'  MAE < 1.0 kg:                 {"PASS" if rep["mae"] < 1.0 else "FAIL"}')
print(f'  Calibration >= 80%:           {"PASS" if rep["calibration_coverage"] >= 0.8 else "FAIL"}')
print(f'  No drift (p > 0.1):           {"PASS" if rep["residual_drift_p"] > 0.1 else "FAIL"}')

MAE:                   1.875 kg
95% calibration:       25.0% of observed weigh-ins inside band
Residual drift p-value: 1.000
Observations used:      8

Phase 1 passing thresholds:
  MAE < 1.0 kg:                 FAIL
  Calibration >= 80%:           FAIL
  No drift (p > 0.1):           PASS


## The three required plots

In [4]:
fig = plotting.trajectory_plot(walk, metric='weight')
plt.show()

In [5]:
fig = plotting.trajectory_plot(walk, metric='bf')
plt.show()

In [6]:
fig = plotting.residual_plot(walk)
plt.show()

## Diagnostic interpretation

If any of the passing thresholds above FAILED, the residual plot is the diagnostic to read. Specifically:

- **Systematic positive residual (observed > predicted everywhere)**: intake is being under-reported. `intake_bias` (Phase 2) will compensate.
- **Monotonic drift over time**: adaptive thermogenesis isn't capturing prolonged-deficit metabolic adaptation. Could motivate Phase 3 NEAT_response.
- **High-variance residuals correlated with carb-heavy days**: glycogen-water dynamics need tuning.
- **Spikes correlated with sodium-heavy days**: sodium-water dynamics need tuning.

Note any patterns here as starting hypotheses for Phase 2.